# Whisper → RKNN Konvertierung für RK3588

Dieses Notebook konvertiert OpenAI Whisper Modelle in das RKNN-Format für die RK3588 NPU.

## Überblick

Whisper besteht aus zwei Teilen die wir separat konvertieren:
- **Encoder**: Verarbeitet das Mel-Spektrogramm (Audio) → interne Audio-Features
- **Decoder**: Generiert Text Token für Token aus den Audio-Features

Beide werden zuerst nach **ONNX** exportiert (universelles ML-Format),
dann mit **rknn-toolkit2** in das RK3588-native **RKNN**-Format konvertiert.

## Warum FP32 statt INT8?

INT8-Quantisierung ist zwar schneller, produziert bei Whisper aber häufig
leere oder fehlerhafte Transkriptionen. FP32 ist stabiler und für unseren
Anwendungsfall (Rezept-Extraktion) ausreichend schnell.

## Ablauf

1. **Dependencies installieren** → Runtime neu starten
2. **Konfiguration** (Modell wählen, GitHub Token eingeben)
3. **Whisper Modell laden**
4. **Encoder → ONNX** exportieren
5. **Decoder → ONNX** exportieren
6. **Encoder ONNX → RKNN** konvertieren
7. **Decoder ONNX → RKNN** konvertieren
8. **Upload** zu GitHub Release

## Schritt 1: Dependencies installieren

> ⚠️ **Nach dieser Zelle: Runtime → Restart session**
> 
> Danach ab Schritt 2 weiterlaufen lassen.
> Der Restart ist nötig damit NumPy korrekt geladen wird
> (rknn-toolkit2 und openai-whisper haben unterschiedliche NumPy-Anforderungen).

In [ ]:
import subprocess, sys

# openai-whisper: wird für Audio-Preprocessing und Tokenizer benötigt
# onnx 1.18.0: letzte Version die noch onnx.mapping hat (von rknn-toolkit2 benötigt)
# onnxruntime: wird für ONNX-Validierung benötigt
# requests: wird für den GitHub Release Upload benötigt
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'openai-whisper', 'onnx==1.18.0', 'onnxruntime', 'requests'])

# rknn-toolkit2: Rockchips Konvertierungs-Toolkit (x86, nur für Konvertierung nicht für Inferenz)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'rknn-toolkit2'])

# NumPy auf kompatible Version fixieren (muss nach rknn-toolkit2 kommen)
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    '--force-reinstall', 'numpy==1.26.4'])

import numpy as np, onnx
print(f'NumPy : {np.__version__}')
print(f'ONNX  : {onnx.__version__}')
print()
print('>>> Jetzt: Runtime -> Restart session')
print('>>> Danach ab Schritt 2 (Konfiguration) weiterlaufen lassen')

## Schritt 2: Konfiguration

Modell auswählen und GitHub Token eingeben.
Der Token wird über eine sichere Eingabe abgefragt und **nicht im Notebook gespeichert**.

In [ ]:
from getpass import getpass
import os

# Whisper Modellgröße wählen
# tiny  ~39MB  - sehr schnell, weniger genau
# base  ~74MB  - guter Kompromiss (empfohlen)
# small ~244MB - genauer, langsamer
# medium~769MB - sehr genau, deutlich langsamer
WHISPER_MODEL   = 'base'

# Ziel-Plattform (RK3588 = RK1, Rock 5B, Orange Pi 5 etc.)
TARGET_PLATFORM = 'rk3588'

# GitHub Release Einstellungen
GITHUB_REPO  = 'Daywalker91/docker-images'
RELEASE_TAG  = 'whisper-rknn-models'
RELEASE_NAME = 'Whisper RKNN Models (RK3588)'

# Ausgabeverzeichnisse
ONNX_DIR = '/tmp/whisper_onnx'
RKNN_DIR = '/tmp/whisper_rknn'
os.makedirs(ONNX_DIR, exist_ok=True)
os.makedirs(RKNN_DIR, exist_ok=True)

# GitHub Token sicher abfragen (wird nicht gespeichert)
# Benötigt: Fine-grained PAT mit 'Contents' (Read & Write)
GITHUB_TOKEN = getpass('GitHub Token (PAT mit Contents Read+Write): ')

print(f'Modell    : whisper-{WHISPER_MODEL}')
print(f'Plattform : {TARGET_PLATFORM}')
print(f'Release   : {GITHUB_REPO} / {RELEASE_TAG}')
print(f'Token     : {"gesetzt" if GITHUB_TOKEN else "FEHLT!"}')

## Schritt 3: Whisper Modell laden

Das Modell wird von OpenAIs Servern heruntergeladen und in den Speicher geladen.
Wir benötigen es nur für den ONNX-Export, nicht für die Inferenz.

In [ ]:
import torch
import whisper

print(f'Lade Whisper-{WHISPER_MODEL}...')
model = whisper.load_model(WHISPER_MODEL, device='cpu')
model.eval()

# Modell-Dimensionen für spätere ONNX-Exports benötigt
n_audio_state = model.dims.n_audio_state  # versteckte Dimension (base=512)
n_text_ctx    = model.dims.n_text_ctx     # max. Token-Länge (alle Modelle=448)

print(f'\nModell-Dimensionen:')
print(f'  n_audio_state : {n_audio_state}  (Encoder Output-Größe)')
print(f'  n_text_ctx    : {n_text_ctx}   (Max. Decoder Token-Länge)')
print(f'\nModell geladen')

## Schritt 4: Encoder → ONNX

Der Encoder nimmt ein Mel-Spektrogramm (80 Frequenzbänder x 3000 Zeitschritte)
und gibt Audio-Features (1500 x n_audio_state) aus.

Die Input-Shape ist bei allen Whisper-Modellen identisch: `(1, 80, 3000)`

In [ ]:
ENCODER_ONNX = f'{ONNX_DIR}/whisper_encoder_{WHISPER_MODEL}.onnx'

class EncoderWrapper(torch.nn.Module):
    """Schlanker Wrapper um den Whisper-Encoder fuer den ONNX-Export."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
    def forward(self, mel):
        return self.encoder(mel)

enc = EncoderWrapper(model.encoder)
enc.eval()

# Dummy-Input: Mel-Spektrogramm mit Batch-Dimension
# Shape (1, 80, 3000) ist bei allen Whisper-Modellen identisch
dummy_mel = torch.zeros(1, 80, 3000)

print('Exportiere Encoder -> ONNX...')
with torch.no_grad():
    torch.onnx.export(
        enc,
        dummy_mel,
        ENCODER_ONNX,
        input_names=['mel'],
        output_names=['encoder_output'],
        # Batch-Dimension dynamisch lassen (fuer Flexibilitaet)
        dynamic_axes={'mel': {0: 'batch'}},
        opset_version=14,
        do_constant_folding=True,
    )

import os
size_mb = os.path.getsize(ENCODER_ONNX) / 1024 / 1024
print(f'Encoder ONNX gespeichert: {ENCODER_ONNX}')
print(f'Groesse: {size_mb:.1f} MB')

## Schritt 5: Decoder → ONNX

Der Decoder generiert Token für Token den Ausgabetext.
Er nimmt die bisherigen Tokens + den Encoder-Output und gibt Logits (Wahrscheinlichkeiten)
für das nächste Token aus.

**Wichtig:** SDPA (Scaled Dot Product Attention) wird deaktiviert da es beim
ONNX-Tracing einen Tensor statt bool erwartet und zu einem Fehler führt.

**Statische Shapes:** RKNN bevorzugt feste Input-Dimensionen. Wir verwenden
die maximale Token-Länge (448) als statische Shape.

In [ ]:
DECODER_ONNX = f'{ONNX_DIR}/whisper_decoder_{WHISPER_MODEL}.onnx'

# SDPA deaktivieren - nicht ONNX-kompatibel (fuehrt zu TypeError beim Tracing)
import whisper.model as wm
wm.MultiHeadAttention.use_sdpa = False

class DecoderWrapper(torch.nn.Module):
    """
    Wrapper um den Whisper-Decoder ohne KV-Cache.
    Inputs : tokens (1, 448), encoder_output (1, 1500, n_audio_state)
    Output : logits (1, 448, vocab_size)
    """
    def __init__(self, decoder):
        super().__init__()
        self.decoder = decoder
    def forward(self, tokens, encoder_output):
        return self.decoder(tokens, encoder_output)

dec = DecoderWrapper(model.decoder)
dec.eval()

# Statische Dummy-Inputs mit maximaler Token-Laenge
# n_text_ctx = 448 fuer alle Whisper-Modelle
dummy_tokens  = torch.zeros(1, n_text_ctx, dtype=torch.long)
dummy_enc_out = torch.zeros(1, 1500, n_audio_state)

print(f'Decoder Input Shapes:')
print(f'  tokens         : {list(dummy_tokens.shape)}')
print(f'  encoder_output : {list(dummy_enc_out.shape)}')
print()
print('Exportiere Decoder -> ONNX...')

with torch.no_grad():
    torch.onnx.export(
        dec,
        (dummy_tokens, dummy_enc_out),
        DECODER_ONNX,
        input_names=['tokens', 'encoder_output'],
        output_names=['logits'],
        # Keine dynamischen Axes - RKNN benoetigt statische Shapes
        opset_version=14,
        do_constant_folding=True,
    )

size_mb = os.path.getsize(DECODER_ONNX) / 1024 / 1024
print(f'Decoder ONNX gespeichert: {DECODER_ONNX}')
print(f'Groesse: {size_mb:.1f} MB')

## Schritt 6 & 7: ONNX → RKNN (FP32)

Beide Modelle werden jetzt mit rknn-toolkit2 in das RKNN-Format konvertiert.

**FP32 (keine Quantisierung):** INT8-Quantisierung wäre schneller, produziert
bei Whisper aber häufig leere oder fehlerhafte Transkriptionen. FP32 ist stabiler.

**input_size_list:** RKNN benötigt explizite Input-Shapes da keine dynamischen
Dimensionen unterstützt werden.

In [ ]:
from rknn.api import RKNN

ENCODER_RKNN = f'{RKNN_DIR}/whisper_encoder_{WHISPER_MODEL}_{TARGET_PLATFORM}.rknn'
DECODER_RKNN = f'{RKNN_DIR}/whisper_decoder_{WHISPER_MODEL}_{TARGET_PLATFORM}.rknn'

def convert_to_rknn(onnx_path, rknn_path, input_names, input_shapes, label):
    """
    Konvertiert ein ONNX-Modell zu RKNN (FP32).
    input_names  : Liste der Input-Namen (muss mit ONNX-Export uebereinstimmen)
    input_shapes : Liste der Input-Shapes [[batch, ...], ...]
    """
    print(f'Konvertiere {label} -> RKNN (FP32)...')

    rknn = RKNN(verbose=False)

    # target_platform: rk3588 = RK1, Rock 5B, Orange Pi 5 etc.
    rknn.config(target_platform=TARGET_PLATFORM)

    # input_size_list: explizite Shapes da RKNN keine dynamischen Dims unterstuetzt
    ret = rknn.load_onnx(
        model=onnx_path,
        inputs=input_names,
        input_size_list=input_shapes
    )
    assert ret == 0, f'load_onnx fehlgeschlagen (Code {ret})'

    # do_quantization=False: FP32 beibehalten (kein INT8)
    ret = rknn.build(do_quantization=False)
    assert ret == 0, f'build fehlgeschlagen (Code {ret})'

    ret = rknn.export_rknn(rknn_path)
    assert ret == 0, f'export_rknn fehlgeschlagen (Code {ret})'

    rknn.release()

    size_mb = os.path.getsize(rknn_path) / 1024 / 1024
    print(f'{label} RKNN gespeichert: {rknn_path}')
    print(f'Groesse: {size_mb:.1f} MB')
    print()

# Encoder: Input ist Mel-Spektrogramm (1, 80, 3000)
convert_to_rknn(
    ENCODER_ONNX, ENCODER_RKNN,
    input_names=['mel'],
    input_shapes=[[1, 80, 3000]],
    label='Encoder'
)

# Decoder: Inputs sind Token-Sequenz und Encoder-Output
convert_to_rknn(
    DECODER_ONNX, DECODER_RKNN,
    input_names=['tokens', 'encoder_output'],
    input_shapes=[[1, n_text_ctx], [1, 1500, n_audio_state]],
    label='Decoder'
)

print('Konvertierung abgeschlossen!')
print(f'  {ENCODER_RKNN}')
print(f'  {DECODER_RKNN}')

## Schritt 8: Upload zu GitHub Release

Die fertigen `.rknn` Dateien werden als Assets zum GitHub Release `whisper-rknn-models` hochgeladen.
Falls der Release noch nicht existiert wird er automatisch erstellt.
Falls bereits Assets mit gleichem Namen vorhanden sind werden diese zuerst gelöscht.

In [ ]:
import requests

assert GITHUB_TOKEN, 'GITHUB_TOKEN ist leer! Bitte Schritt 2 erneut ausfuehren.'

headers  = {
    'Authorization': f'Bearer {GITHUB_TOKEN}',
    'Accept': 'application/vnd.github+json',
    'X-GitHub-Api-Version': '2022-11-28'
}
api_base = f'https://api.github.com/repos/{GITHUB_REPO}'

# Release suchen oder neu erstellen
print(f'Suche Release "{RELEASE_TAG}"...')
r = requests.get(f'{api_base}/releases/tags/{RELEASE_TAG}', headers=headers)

if r.status_code == 200:
    release_id = r.json()['id']
    upload_url = r.json()['upload_url'].replace('{?name,label}', '')
    print(f'Release gefunden (ID: {release_id})')

    # Bestehende Assets mit gleichem Namen loeschen (Ueberschreiben)
    existing = requests.get(f'{api_base}/releases/{release_id}/assets', headers=headers).json()
    target_files = [os.path.basename(ENCODER_RKNN), os.path.basename(DECODER_RKNN)]
    for asset in existing:
        if asset['name'] in target_files:
            requests.delete(f'{api_base}/releases/assets/{asset["id"]}', headers=headers)
            print(f'  Altes Asset geloescht: {asset["name"]}')
else:
    # Release existiert noch nicht - erstellen
    print('Release nicht gefunden - wird erstellt...')
    r = requests.post(
        f'{api_base}/releases',
        headers=headers,
        json={
            'tag_name':   RELEASE_TAG,
            'name':       RELEASE_NAME,
            'body':       f'Konvertierte Whisper RKNN Modelle fuer {TARGET_PLATFORM.upper()} (FP32, keine Quantisierung).\nModell: whisper-{WHISPER_MODEL}',
            'prerelease': True,
        }
    )
    assert r.status_code == 201, f'Release-Erstellung fehlgeschlagen: {r.text}'
    release_id = r.json()['id']
    upload_url = r.json()['upload_url'].replace('{?name,label}', '')
    print(f'Release erstellt (ID: {release_id})')

# RKNN Dateien hochladen
print()
for filepath in [ENCODER_RKNN, DECODER_RKNN]:
    filename = os.path.basename(filepath)
    size_mb  = os.path.getsize(filepath) / 1024 / 1024
    print(f'Uploade {filename} ({size_mb:.1f} MB)...')

    with open(filepath, 'rb') as f:
        r = requests.post(
            f'{upload_url}?name={filename}',
            headers={**headers, 'Content-Type': 'application/octet-stream'},
            data=f
        )

    if r.status_code == 201:
        print(f'  OK: {r.json()["browser_download_url"]}')
    else:
        print(f'  Fehler ({r.status_code}): {r.text}')

print()
print(f'Fertig! https://github.com/{GITHUB_REPO}/releases/tag/{RELEASE_TAG}')